# Telugu OCR using GOT-OCR2.0 and MS-SWIFT

In [ ]:
!nvidia-smi

Thu Jun 25 06:10:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


First, we mount Google Drive to access our dataset and project directory.

In [ ]:
!ls /content/drive/MyDrive

'Colab Notebooks'   COURSERA_NOTES.gdoc   IIIT-HW-Telugu.zip   Telugu_GOT_OCR


We then copy and unzip the dataset containing Telugu handwritten text images.

In [ ]:

!cp "/content/drive/MyDrive/IIIT-HW-Telugu.zip" /content/IIIT-HW-Telugu.zip


!unzip -q /content/IIIT-HW-Telugu.zip -d /content/


!unzip -q -o /content/IIIT-HW-Telugu/TeluguSeg.zip -d /content/IIIT-HW-Telugu/

In [ ]:
!ls /content/IIIT-HW-Telugu

lexicon.txt  TeluguSeg	    telugu_vocab.txt  train.txt
Readme.txt   TeluguSeg.zip  test.txt	      val.txt


Let's inspect the contents of the unzipped directory to understand the file structure.

In [ ]:
with open('/content/IIIT-HW-Telugu/train.txt','r',encoding='utf-8') as f:
    for i in range(10):
        print(f.readline())

TeluguSeg/train/3/259/1.jpg పర్వతాన్ని

TeluguSeg/train/3/259/2.jpg కలుషితమై

TeluguSeg/train/3/259/3.jpg కంట్రాక్టర్లు

TeluguSeg/train/3/259/4.jpg తీసుకోబోతున్నాడని

TeluguSeg/train/3/259/5.jpg సంరక్షకులుగా

TeluguSeg/train/3/259/6.jpg కవర్లు

TeluguSeg/train/3/259/7.jpg తాకడం

TeluguSeg/train/3/259/9.jpg పదుకొనె

TeluguSeg/train/3/259/10.jpg చెప్పాయి

TeluguSeg/train/3/259/11.jpg ప్రతివ్యూహాలు



We'll check the format of the `train.txt` file, which contains image paths and corresponding text labels.

In [ ]:
import os

img = "/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg"

print(os.path.exists(img))

True


Verify if a sample image path extracted from `train.txt` actually exists on the filesystem.

In [ ]:
path, text = line.split(maxsplit=1)

print("PATH:", path)
print("TEXT:", text)

PATH: TeluguSeg/train/3/259/1.jpg
TEXT: పర్వతాన్ని


Demonstrate how an individual line from the text file is split into an image path and its corresponding text.

In [ ]:
def count_lines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

print("Train:", count_lines('/content/IIIT-HW-Telugu/train.txt'))
print("Val:", count_lines('/content/IIIT-HW-Telugu/val.txt'))
print("Test:", count_lines('/content/IIIT-HW-Telugu/test.txt'))

Train: 80637
Val: 19980
Test: 17898


Count the number of samples in the training, validation, and test datasets to understand their sizes.

In [ ]:
with open('/content/IIIT-HW-Telugu/train.txt', 'r', encoding='utf-8') as f:
    for i in range(5):
        print(f.readline().strip())

TeluguSeg/train/3/259/1.jpg పర్వతాన్ని
TeluguSeg/train/3/259/2.jpg కలుషితమై
TeluguSeg/train/3/259/3.jpg కంట్రాక్టర్లు
TeluguSeg/train/3/259/4.jpg తీసుకోబోతున్నాడని
TeluguSeg/train/3/259/5.jpg సంరక్షకులుగా


Display the first few lines of the `train.txt` file again to confirm its structure.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Telugu_GOT_OCR"

JSON_DIR = os.path.join(PROJECT_DIR, "jsons")

os.makedirs(JSON_DIR, exist_ok=True)

print(JSON_DIR)

/content/drive/MyDrive/Telugu_GOT_OCR/jsons


Define the project directory and a `jsons` subdirectory where processed data will be stored.

In [ ]:
import json
import os

def create_swift_got_json(txt_file, output_json, max_samples=None):
    data = []

    with open(txt_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if max_samples and idx >= max_samples:
                break

            line = line.strip()
            if not line:
                continue


            image_rel, text = line.split(maxsplit=1)


            image_abs = os.path.join("/content/IIIT-HW-Telugu", image_rel)


            sample = {
                "query": "<image>\nOCR:",
                "response": text.strip(),
                "images": [image_abs]
            }
            data.append(sample)

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Saved {len(data)} samples -> {output_json}")


create_swift_got_json(
    "/content/IIIT-HW-Telugu/train.txt",
    f"{JSON_DIR}/train_small.json",
    max_samples=5000
)

create_swift_got_json(
    "/content/IIIT-HW-Telugu/val.txt",
    f"{JSON_DIR}/val_small.json",
    max_samples=1000
)

create_swift_got_json(
    "/content/IIIT-HW-Telugu/test.txt",
    f"{JSON_DIR}/test_small.json",
    max_samples=1000
)

Saved 5000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.json
Saved 1000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.json
Saved 1000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.json


The `create_swift_got_json` function converts the raw `.txt` dataset files into a JSON format suitable for the MS-SWIFT framework. This initial run generates small subsets (5k train, 1k val, 1k test) for quicker initial experiments.

In [ ]:
!ls "/content/drive/MyDrive/Telugu_GOT_OCR/jsons"

test_100.jsonl	  train_full.json      train_small.jsonl  val_small.json
test_full.json	  train_full.jsonl     val_2k.json	  val_small.jsonl
test_full.jsonl   train_small_2.json   val_2k.jsonl
test_small.json   train_small_2.jsonl  val_small_2.json
test_small.jsonl  train_small.json     val_small_2.jsonl


List the contents of the `jsons` directory to confirm the creation of the new JSON files.

In [ ]:
import json

with open(
    f"{JSON_DIR}/train_small.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(data[0])

{'query': '<image>\nOCR:', 'response': 'పర్వతాన్ని', 'images': ['/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg']}


Load and display a sample from the generated `train_small.json` to inspect its structure.

In [ ]:
!ls "/content/drive/MyDrive/Telugu_GOT_OCR/jsons"

test_100.jsonl	  train_full.json      train_small.jsonl  val_small.json
test_full.json	  train_full.jsonl     val_2k.json	  val_small.jsonl
test_full.jsonl   train_small_2.json   val_2k.jsonl
test_small.json   train_small_2.jsonl  val_small_2.json
test_small.jsonl  train_small.json     val_small_2.jsonl


Re-list the `jsons` directory contents.

In [ ]:
import json

def json_to_jsonl(json_file, jsonl_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    with open(jsonl_file, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved {len(data)} samples to {jsonl_file}")

The `json_to_jsonl` function converts the JSON files into JSONL (JSON Lines) format, which is often preferred for streaming data processing in machine learning pipelines.

In [ ]:
json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl"
)

  json_to_jsonl(
      "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.json",
      "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"
  )

Saved 5000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl
Saved 1000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl
Saved 1000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl


Convert the previously generated small JSON datasets into JSONL format.

In [ ]:
with open(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for i in range(3):
        print(f.readline())

{"query": "<image>\nOCR:", "response": "పర్వతాన్ని", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg"]}

{"query": "<image>\nOCR:", "response": "కలుషితమై", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/2.jpg"]}

{"query": "<image>\nOCR:", "response": "కంట్రాక్టర్లు", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/3.jpg"]}



Display the first few lines of the generated `train_small.jsonl` to confirm the format.

In [ ]:
# 1. Install the MS-SWIFT wrapper framework
!pip install "ms-swift[llm]" -U

# 2. Install supporting image decoding and training orchestration packages
!pip install timm transformers accelerate decord einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: datasets
    Found existing installation: datasets 2.21.

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
^C


## Environment Setup: Installing Dependencies

In [ ]:
!pip install decord

In [ ]:
!pip install verovio

In [ ]:
!pip show ms-swift

Name: ms_swift
Version: 4.3.1
Summary: Swift: Scalable lightWeight Infrastructure for Fine-Tuning
Home-page: https://github.com/modelscope/ms-swift
Author: DAMO ModelScope teams
Author-email: contact@modelscope.cn
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, addict, aiohttp, attrdict, binpacking, charset-normalizer, cpm-kernels, dacite, datasets, einops, fastapi, gradio, importlib-metadata, json-repair, matplotlib, modelscope, nltk, numpy, openai, oss2, pandas, peft, pillow, PyYAML, requests, rouge, safetensors, scipy, sentencepiece, simplejson, sortedcontainers, tensorboard, tiktoken, tqdm, transformers, transformers-stream-generator, trl, uvicorn, zstandard
Required-by: 


Verify the installed version of `ms-swift` to ensure compatibility.

Install `torchao`. This is likely for optimizations related to PyTorch operations.

In [ ]:
!pip install "torchao>=0.16.0"

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2 \
    --num_train_epochs 3 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --eval_strategy epoch

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2 --num_train_epochs 3 --per_device_train_batch_size 1 --gradient_accumulation_steps 16 --learning_rate 5e-5 --max_length 512 --save_strategy steps --save_steps 150 --save_total_limit 5 --eval_strategy epoch`
Failed to load /usr/local/lib/python3.12/dist-packages/t

Initiate training of the `GOT-OCR2_0` model using the `swift sft` command. This run uses the small training and validation datasets (5k/1k samples respectively) for 3 epochs.

## Model Training: Initial Run (Small Dataset)

In [ ]:
 !pip install "datasets==2.21.0"

Install a specific version of the `datasets` library to ensure compatibility with the training environment.

In [ ]:
import datasets
print("datasets version:", datasets.__version__)

datasets version: 2.21.0


Verify the installed `datasets` version.

##Testing checkpoint-500 and analyzing the evaluation metrics ans accuracy

In [ ]:
import os

CHECKPOINT = "/content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939"

print(os.path.exists(CHECKPOINT))
print(os.listdir(CHECKPOINT))

True
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'additional_config.json', 'training_args.bin', 'args.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']


Before evaluation, we need to locate the checkpoint files and confirm their existence. This block checks for the presence of `checkpoint-939`.

## Model Evaluation: Checkpoint 939 (Small Test Set)

In [ ]:
TEST_FILE = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"

print(os.path.exists(TEST_FILE))

True


Confirm the existence of the full test dataset JSONL file.

In [ ]:
import json

input_file = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"
output_file = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(output_file, "w", encoding="utf-8") as f:
    f.writelines(lines[:100])

print("Saved", len(lines[:100]), "samples")

Saved 100 samples


To quickly evaluate the model, we create a smaller test set (`test_100.jsonl`) containing only the first 100 samples from the full test set.

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 \
    --infer_backend transformers \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint500.jsonl

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 --infer_backend transformers --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint500.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C

Run inference using the `swift infer` command with `checkpoint-939` on the `test_100.jsonl` dataset. The results will be saved to `results_checkpoint500.jsonl`.

In [ ]:
!pip install transformers==4.51.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.29.1 requires transformers>=4.56.2, but you have transformers 4.51.3 which is incompatible.
litellm 1.88.1 requires httpx<1.0,>=0.28.0, but you have httpx 0.27.2 which is incompatible.


Install a specific version of the `transformers` library, which is crucial for model compatibility. The output also highlights potential dependency conflicts.

#Evaluation of the checkpoint-500

In [ ]:
!pip install jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.1 MB/s eta 0:00:00


Install the `jiwer` library for calculating Character Error Rate (CER) and Word Error Rate (WER).

## Evaluation Metrics: Checkpoint 939 (100 Test Samples)

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint500.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.7621
WER: 1.0000


Load the inference results from `results_checkpoint500.jsonl` and calculate the Character Error Rate (CER) and Word Error Rate (WER) to assess model performance.

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 23.79%


Calculate the character accuracy based on the overall CER.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 0.00%


Calculate the exact match accuracy, which is the percentage of predictions that perfectly match the ground truth.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : మ్యిపోంటో
--------------------------------------------------
GT   : పదులు
Pred : మ్మి
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : తుల్చిచిటటిచు
--------------------------------------------------
GT   : హేమచంద్ర
Pred : చ్ష్యిచంచే
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : సంట్యించిందు
--------------------------------------------------
GT   : కృషికి
Pred : క్యాక
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చుంట్తుమేస్చో
--------------------------------------------------
GT   : లేదంటోంది
Pred : వచండాంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : అరిమ్స్ట్
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : పీతులులటటు
--------------------------------------------------
GT   : మెరీనా
Pred : మూర్స్స్
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కెటంచాన్నో
------------------------

Display some examples of mispredicted samples to understand common errors made by the model.

##Evaluation of checkpoint - 939

## Model Evaluation: Checkpoint 939 (100 Test Samples - Re-run)

In [ ]:
import os

CHECKPOINT = "/content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939"

print(os.path.exists(CHECKPOINT))
print(os.listdir(CHECKPOINT))

True
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'additional_config.json', 'training_args.bin', 'args.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']


Confirm the existence and contents of `checkpoint-939` again.

In [ ]:
TEST_FILE = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"

print(os.path.exists(TEST_FILE))

True


Confirm the existence of the full test dataset JSONL file.

In [ ]:
import json

input_file = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"
output_file = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(output_file, "w", encoding="utf-8") as f:
    f.writelines(lines[:100])

print("Saved", len(lines[:100]), "samples")

Saved 100 samples


Re-create the `test_100.jsonl` file to ensure a consistent test set.

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-17 19:30:29.300884: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical o

Run inference with `checkpoint-939` on the `test_100.jsonl` dataset. This time, `infer_backend` is set to `pt` (PyTorch) instead of `transformers`.

In [ ]:
pip install transformers==4.51.3

Install `transformers==4.51.3` again, as it might have been uninstalled or updated by other processes.

In [ ]:
import glob

paths = glob.glob("/root/.cache/huggingface/modules/transformers_modules/**/modeling_GOT.py", recursive=True) + \
        glob.glob("/root/.cache/modelscope/hub/models/**/modeling_GOT.py", recursive=True)

print("Found:", paths)

for path in paths:
    with open(path, "r") as f:
        text = f.read()

    changed = False

    if "past_length = past_key_values.seen_tokens" in text:
        text = text.replace(
            "past_length = past_key_values.seen_tokens",
            "past_length = cache_length"
        )
        changed = True

    if "get_max_length()" in text:
        text = text.replace("get_max_length()", "get_seq_length()")
        changed = True

    if changed:
        with open(path, "w") as f:
            f.write(text)
        print(f"✅ Patched: {path}")
    else:
        print(f"Already patched or no match: {path}")

Found: ['/root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py', '/root/.cache/modelscope/hub/models/stepfun-ai/GOT-OCR2_0/modeling_GOT.py']
✅ Patched: /root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py
✅ Patched: /root/.cache/modelscope/hub/models/stepfun-ai/GOT-OCR2_0/modeling_GOT.py


Apply necessary patches to `modeling_GOT.py` files. This is a common step to resolve compatibility issues or bugs in model implementations from Hugging Face or ModelScope caches, specifically related to `past_length` and `get_max_length`.

In [ ]:
!grep -n "cache_length" /root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py

399:                cache_length = past_key_values.get_seq_length()
400:                past_length = cache_length
401:                max_cache_length = past_key_values.get_seq_length()
403:                cache_length = past_length = past_key_values[0][0].shape[2]
404:                max_cache_length = None
420:                max_cache_length is not None
422:                and cache_length + input_ids.shape[1] > max_cache_length
424:                attention_mask = attention_mask[:, -max_cache_length:]


Verify that `cache_length` is correctly applied in the patched `modeling_GOT.py` file.

In [ ]:
!grep -n "get_seq_length" /root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py

399:                cache_length = past_key_values.get_seq_length()
401:                max_cache_length = past_key_values.get_seq_length()


Verify that `get_seq_length` is correctly applied in the patched `modeling_GOT.py` file.

##Calculation of evaluation metrics on checkpoint-939

## Evaluation Metrics: Checkpoint 939 (Re-run on 100 Samples)

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.6386
WER: 1.0000


Re-calculate CER and WER for `checkpoint-939` using the results from the `pt` backend inference.

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 36.14%


Re-calculate character accuracy.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 0.00%


Re-calculate exact match accuracy.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : వేస్పేయంలో
--------------------------------------------------
GT   : పదులు
Pred : మికలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : తుల్కావికలాచ్చు
--------------------------------------------------
GT   : హేమచంద్ర
Pred : హ్రివ్మిచంట్
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : గెంతకుముంద్
--------------------------------------------------
GT   : కృషికి
Pred : క్షిష్
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చుంట్టివున్నీని
--------------------------------------------------
GT   : లేదంటోంది
Pred : విదంతోంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : సారుాన్తే
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : పీటుకులతలలు
--------------------------------------------------
GT   : మెరీనా
Pred : వెందీనా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కొలుంణాన్ని
--------------

Display mispredicted samples from the re-run inference.

##Evaluation of checkpoint-939 on 1000 test images

## Model Evaluation: Checkpoint 939 (Full 1000 Test Images)

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939_2.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939_2.jsonl`
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/swift/cli/infer.py", line 2, in <module>
    from swift.pipelines import infer_main
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/swift/utils/import_utils.py", line 104, in __getattr__
    value = getattr(module, name)
            ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/swift/utils/import_utils.py", line 103, in __getattr__
    module = self._get_module(self._class_to_module[name])
Traceback (

Run inference with `checkpoint-939` on the full `test_small.jsonl` dataset (1000 samples). This might take longer due to the larger dataset size.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939_2.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.6106
WER: 0.9910


Calculate CER and WER for `checkpoint-939` using the results from the full 1000 test images.

## Evaluation Metrics: Checkpoint 939 (Full 1000 Test Images)

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 38.94%


Calculate character accuracy for the full test set.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 0.90%


Calculate exact match accuracy for the full test set.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : వేస్పేయంలో
--------------------------------------------------
GT   : పదులు
Pred : మికలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : తుల్కావికలాచ్చు
--------------------------------------------------
GT   : హేమచంద్ర
Pred : హ్రివ్మిచంట్
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : గెంతకుముంద్
--------------------------------------------------
GT   : కృషికి
Pred : క్షిష్
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చుంట్టివున్నీని
--------------------------------------------------
GT   : లేదంటోంది
Pred : విదంతోంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : సారుాన్తే
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : పీటుకులతలలు
--------------------------------------------------
GT   : మెరీనా
Pred : వెందీనా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కొలుంణాన్ని
--------------

Display mispredicted samples from the full test set evaluation.

## Training the vlm model on increased 10k images training dataset

## Data Preparation: Increased Training Dataset (10k samples)

In [ ]:
import json
import os

def create_swift_got_json(txt_file, output_json, max_samples=None):
    data = []

    with open(txt_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if max_samples and idx >= max_samples:
                break

            line = line.strip()
            if not line:
                continue

            # Split the line by the first whitespace into image path and the text label
            image_rel, text = line.split(maxsplit=1)

            # Form the absolute image path exactly how you did before
            image_abs = os.path.join("/content/IIIT-HW-Telugu", image_rel)

            # Format explicitly for MS-SWIFT's multi-modal structure
            # The <image> tag acts as the token positioning placeholder for the model
            sample = {
                "query": "<image>\nOCR:",
                "response": text.strip(),
                "images": [image_abs]
            }
            data.append(sample)

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Saved {len(data)} samples -> {output_json}")

# --- EXECUTE GENERATION ---
# This matches your exactly planned 5k / 1k / 1k strategy!
create_swift_got_json(
    "/content/IIIT-HW-Telugu/train.txt",
    f"{JSON_DIR}/train_small_2.json",
    max_samples=10000
)

create_swift_got_json(
    "/content/IIIT-HW-Telugu/val.txt",
    f"{JSON_DIR}/val_small_2.json",
    max_samples=2000
)

create_swift_got_json(
    "/content/IIIT-HW-Telugu/test.txt",
    f"{JSON_DIR}/test_small.json",
    max_samples=1000
)

Saved 10000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small_2.json
Saved 2000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small_2.json
Saved 1000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.json


Re-run `create_swift_got_json` to generate larger datasets: 10,000 training samples and 2,000 validation samples, while keeping the test set at 1,000 samples.

In [ ]:
json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small_2.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small_2.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small_2.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small_2.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"
)

Saved 10000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small_2.jsonl
Saved 2000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small_2.jsonl
Saved 1000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl


Convert the newly generated JSON files for the increased datasets into JSONL format.

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small_2.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small_2.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_3 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v3-20260613-160903/checkpoint-939 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small_2.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small_2.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_3 --num_train_epochs 5 --per_device_train_batch_size 4 --gradient_accumulation_steps 8 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not lo

Continue training the `GOT-OCR2_0` model, resuming from `checkpoint-939`, but now using the increased training and validation datasets (10k/2k samples). This run is configured for 5 epochs with adjusted batch size and gradient accumulation.

## Model Training: Increased Dataset (10k Train, 2k Val)

##Evaluation on checkpint-150

## Model Evaluation: Checkpoint 150

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_3/v0-20260618-170023/checkpoint-150 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 2 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint150.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_3/v0-20260618-170023/checkpoint-150 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 2 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint150.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-18 19:33:11.789965: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical

Run inference using `checkpoint-150` from the new training run on the 1000-sample test set. The results are saved to `results_checkpoint150.jsonl`.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint150.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.5632
WER: 0.9780


Calculate CER and WER for `checkpoint-150`.

## Evaluation Metrics: Checkpoint 150

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 43.68%


Calculate character accuracy for `checkpoint-150`.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 2.20%


Calculate exact match accuracy for `checkpoint-150`.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : వేస్షిషేకంలో
--------------------------------------------------
GT   : పదులు
Pred : యదలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : తుల్చాకాహలాచ్చు
--------------------------------------------------
GT   : హేమచంద్ర
Pred : వైవిచిచంట్
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : గెంతకుముంద్
--------------------------------------------------
GT   : కృషికి
Pred : క్షషకి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చుంట్టవిన్ని
--------------------------------------------------
GT   : లేదంటోంది
Pred : విదంటారి
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గెరావాస్తే
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : పీతక్షాలతలు
--------------------------------------------------
GT   : మెరీనా
Pred : వెరిరీనా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కొలుంానాన్నే
----------------

Display mispredicted samples from the `checkpoint-150` evaluation.

## Training the GOT-OCR-2.0 on full dataset

## Data Preparation: Full Training Dataset

In [ ]:
import json
import os
import random

JSON_DIR = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons"

def create_swift_got_json(txt_file, output_json, max_samples=None, shuffle=True):
    data = []

    # Read all lines
    with open(txt_file, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    # Shuffle for better training distribution
    if shuffle:
        random.seed(42)
        random.shuffle(lines)

    # Take subset if requested
    if max_samples is not None:
        lines = lines[:max_samples]

    # Create samples
    for line in lines:
        image_rel, text = line.split(maxsplit=1)

        image_abs = os.path.join(
            "/content/IIIT-HW-Telugu",
            image_rel
        )

        sample = {
            "query": "<image>\nOCR:",
            "response": text.strip(),
            "images": [image_abs]
        }

        data.append(sample)

    # Save JSON
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Saved {len(data)} samples -> {output_json}")


# =========================
# TRAIN (FULL 80,637)
# =========================
create_swift_got_json(
    "/content/IIIT-HW-Telugu/train.txt",
    f"{JSON_DIR}/train_full.json",
    shuffle=True
)

# =========================
# VALIDATION (2,000)
# =========================
create_swift_got_json(
    "/content/IIIT-HW-Telugu/val.txt",
    f"{JSON_DIR}/val_2k.json",
    max_samples=2000,
    shuffle=True
)

# =========================
# TEST (FULL 17,898)
# =========================
create_swift_got_json(
    "/content/IIIT-HW-Telugu/test.txt",
    f"{JSON_DIR}/test_full.json",
    shuffle=False
)

print("\nDataset preparation completed.")

Saved 80637 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.json
Saved 2000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.json
Saved 17898 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_full.json

Dataset preparation completed.


Prepare the full datasets for training, validation, and testing. This includes shuffling the training and validation sets and converting them to the MS-SWIFT JSON format. The training set is the full 80,637 samples.

In [ ]:
import json

def json_to_jsonl(json_file, jsonl_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    with open(jsonl_file, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved {len(data)} samples to {jsonl_file}")

Define the `json_to_jsonl` function again (it was defined earlier but might be helpful to have it explicitly here before its usage).

In [ ]:
json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_full.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_full.jsonl"
)

Saved 80637 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl
Saved 2000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl
Saved 17898 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_full.jsonl


Convert the full training, validation, and test JSON files into JSONL format.

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_3/v0-20260618-170023/checkpoint-150 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 180 \
    --save_strategy steps \
    --save_steps 180 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_3/v0-20260618-170023/checkpoint-150 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 180 --save_strategy steps --save_steps 180 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-

Continue training the `GOT-OCR2_0` model, resuming from `checkpoint-150` (from the previous run on the increased dataset), but now using the *full* training dataset (80,637 samples) and 2,000 validation samples. This training also incorporates `float16` precision and gradient checkpointing for efficiency.

## Model Training: Full Dataset (Continued from Checkpoint 150)

## Continuing Training on whole dataset from checkpoint-1800

## Model Training: Full Dataset (Continued from Checkpoint 1800)

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v1-20260620-121006/checkpoint-1800 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v1-20260620-121006/checkpoint-1800 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/tor

Further continue training the `GOT-OCR2_0` model, resuming from `checkpoint-1800` of the full dataset training. This continues the fine-tuning process on the complete dataset.

## Continuing training from checkpoint-4680

## Model Evaluation: Checkpoint 7200

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v8-20260622-155032/checkpoint-4680 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v8-20260622-155032/checkpoint-4680 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/tor

## Evaluation of checkpoint-7200

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v12-20260623-060835/checkpoint-7200 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 8 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint7200.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v12-20260623-060835/checkpoint-7200 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 8 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint7200.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
2026-06-24 04:22:14.192765: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-

Run inference using `checkpoint-7200` on the 1000-sample test set to evaluate its performance at this stage of training.

In [ ]:
!pip install jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.0 MB/s eta 0:00:00


Install `jiwer` again to ensure it's available for evaluation.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint7200.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.3399
WER: 0.8673


Calculate CER and WER for `checkpoint-7200`.

## Evaluation Metrics: Checkpoint 7200

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 66.01%


Calculate character accuracy for `checkpoint-7200`.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 13.27%


Calculate exact match accuracy for `checkpoint-7200`.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : మసిష్కంలో
--------------------------------------------------
GT   : పదులు
Pred : పదలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : టలోకాదకాలచ్చి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : ప్రివచంద్ర
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : ఇంతకుయుదు
--------------------------------------------------
GT   : కృషికి
Pred : టీషికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చూడివన్నది
--------------------------------------------------
GT   : లేదంటోంది
Pred : వేదంటాంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గారవాన్ని
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : ప్రతకాలతలు
--------------------------------------------------
GT   : మెరీనా
Pred : మెరిగా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కుటించాన్ని
-----------------------------

Display mispredicted samples from the `checkpoint-7200` evaluation.

## continuing the training on whole training dataset from checkpoint-7200

## Model Training: Full Dataset (Continued from Checkpoint 7200)

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v12-20260623-060835/checkpoint-7200 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v12-20260623-060835/checkpoint-7200 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_9

Continue training the `GOT-OCR2_0` model, resuming from `checkpoint-7200`. This further fine-tunes the model on the full dataset with the specified parameters.

## Evaluating on checkpoint-10980

## Model Evaluation: Checkpoint 10980

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v21-20260624-181828/checkpoint-10980 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 8 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint10980.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v21-20260624-181828/checkpoint-10980 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 8 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint10980.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-25 06:28:08.953849: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performanc

Run inference using `checkpoint-10980` on the 1000-sample test set. This evaluates the model's performance after more extensive training.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint10980.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.3787
WER: 0.8790


Calculate CER and WER for `checkpoint-10980`.

## Evaluation Metrics: Checkpoint 10980

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 62.13%


Calculate character accuracy for `checkpoint-10980`.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 12.10%


Calculate exact match accuracy for `checkpoint-10980`.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : యస్తియంలో
--------------------------------------------------
GT   : పదులు
Pred : పేదలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : అలోకాదాలచ్చి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : ప్రిమచంద్ర
--------------------------------------------------
GT   : కృషికి
Pred : ప్రషికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చూడివన్నది
--------------------------------------------------
GT   : లేదంటోంది
Pred : వేదంటాంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గారవాన్ని
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : ప్రతికాలతలు
--------------------------------------------------
GT   : మెరీనా
Pred : మెరిగా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కుటించాన్ని
--------------------------------------------------
GT   : తర్వాతయినా
Pred : ఆంప్యాలోనా
--------------------------

Display mispredicted samples from the `checkpoint-10980` evaluation.

## Evaluating on checkpoint-11160

## Model Evaluation: Checkpoint 11160

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint10980.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.3684
WER: 0.8715


Calculate CER and WER using the results from `results_checkpoint10980.jsonl` (there might be a typo in the cell description, as it states 'checkpoint-11160' but uses the 10980 results file).

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v21-20260624-181828/checkpoint-11160 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_full.jsonl \
    --max_batch_size 16\
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint11160.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v21-20260624-181828/checkpoint-11160 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_full.jsonl --max_batch_size 16 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint11160.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-25 06:51:31.212335: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performanc

Run inference using `checkpoint-11160` on the *full* `test_full.jsonl` dataset (17,898 samples). This is a comprehensive evaluation of the model's final performance.

## Continuing training on whole dataset from checkpoint-11160

## Model Training: Full Dataset (Continued from Checkpoint 11160)

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v21-20260624-181828/checkpoint-11160 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v21-20260624-181828/checkpoint-11160 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_full.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/t

Continue training the `GOT-OCR2_0` model, resuming from `checkpoint-11160` to further improve its performance on the complete training dataset.